# Converting the SymbTr v3.0 TXT Corpus into a MusicXML Dataset

## Overview

This notebook converts the SymbTr Turkish makam music corpus from its original symbolic TXT representation into a standardized MusicXML dataset.

The SymbTr corpus stores each musical composition as a structured TXT file containing symbolic musical events together with musical metadata. Although these TXT files are suitable for computational processing, they are specific to the SymbTr format and are not directly compatible with many music notation software packages or symbolic music processing libraries.

To improve interoperability and facilitate subsequent computational analyses, every TXT file is converted into an individual MusicXML document. During this process, the notebook preserves both the musical content and the available metadata, including note sequences, rhythmic information, lyrics, and descriptive information such as the composition title, makam, usul, musical form, and composer whenever available.

Each generated MusicXML file is validated before being stored in the output directory, ensuring that the resulting corpus forms a reliable and reproducible symbolic music dataset. The exported MusicXML collection serves as a standardized representation of the Turkish makam corpus and can be used in subsequent notebooks for symbolic music analysis, machine learning, and AI-based music generation.

The conversion workflow consists of the following stages:

1. Locate all SymbTr TXT files.
2. Parse each TXT file into a structured DataFrame.
3. Extract musical metadata from the filename.
4. Convert symbolic musical events into MusicXML elements.
5. Preserve musical attributes, including pitch, octave, microtonal alterations, durations, lyrics, and measure information.
6. Validate the generated MusicXML document.
7. Save each composition as an individual MusicXML file.
8. Generate a complete MusicXML corpus for subsequent computational music analysis.
The conversion pipeline consists of the following stages:


TXT parsing
+
Symbolic information extraction
->

MusicXML generation


Each TXT file represents a single musical composition. During the conversion process, symbolic musical events are parsed and transformed into MusicXML elements while preserving essential musical information.

The conversion preserves:

- Pitch information
- Microtonal pitch characteristics
- Rhythmic information
- Lyric information
- Makam metadata
- Musical form
- Usul information
- Composer information

The final output of this notebook is a complete MusicXML dataset containing **3,000 Turkish makam music compositions**, where each original SymbTr TXT file is represented as an individual MusicXML document.



## Preservation of Turkish Makam Characteristics

Turkish makam music contains microtonal pitch structures that cannot be fully represented using conventional twelve-tone equal temperament systems.

Therefore, the conversion process preserves SymbTr microtonal information by mapping the `KomaAE` values into the MusicXML `<alter>` element.

This enables the generated dataset to maintain makam-specific melodic characteristics and provides AI models with access to the pitch deviations required for learning traditional Turkish makam structures.

Example:

```xml
<pitch>
    <step>C</step>
    <alter>1.59</alter>
    <octave>5</octave>
</pitch>

# Converting SymbTr V3.0 TXT Corpus into MusicXML Dataset

This notebook prepares the SymbTr symbolic music corpus for AI-based music generation.

The original corpus is provided in TXT format containing symbolic Turkish makam music notation. Since many AI music generation frameworks require standardized symbolic formats, the TXT files are converted into MusicXML representations.

The conversion pipeline consists of the following stages:

1. Downloading the SymbTr TXT corpus
2. Extracting and organizing TXT files
3. Reading symbolic music representations
4. Parsing pitch, duration, and structural information
5. Converting symbolic events into MusicXML format
6. Preparing the generated XML files for AI music generation experiments


Each TXT file represents a single musical composition. During the conversion process, every symbolic composition was transformed into an individual MusicXML document while preserving important musical attributes.

The generated MusicXML dataset contains **3,000 symbolic music compositions** and provides an AI-compatible representation of Turkish makam music.

The conversion preserves:

- Pitch information
- Microtonal pitch characteristics
- Rhythmic structures
- Lyric information
- Makam metadata
- Musical form
- Usul information
- Composer information

The resulting MusicXML corpus serves as the foundation for subsequent AI-based symbolic music analysis, sequence modeling, and Turkish makam music generation experiments.

## Import Required Libraries

This section imports the Python libraries required for the SymbTr TXT-to-MusicXML conversion pipeline.

The conversion process requires libraries for file system management, dataset downloading, compressed file extraction, XML generation, and progress monitoring.

The imported libraries provide the following functionalities:

- **Pathlib**: Provides platform-independent path management for organizing dataset directories.
- **Zipfile**: Enables extraction of compressed SymbTr dataset archives.
- **Requests**: Allows automatic downloading of the original TXT corpus from the online data repository.
- **Shutil**: Supports file transfer operations during dataset preparation.
- **xml.etree.ElementTree**: Provides XML document generation capabilities for creating MusicXML files.
- **Tqdm**: Displays progress information during large-scale conversion of multiple TXT files.

These libraries establish the computational environment required for transforming symbolic Turkish makam music representations into a standardized XML-based format suitable for AI music generation experiments.


---

## Musical Metadata Representation

The original SymbTr file naming convention contains important information about each composition.

For example:


is converted into the following metadata:

| Attribute | Value |
|---|---|
| Makam | Acem |
| Form | İlahi |
| Usul | Düyek |
| Composition | Aldanma Dunya |
| Composer | Zekai Dede |

These attributes are embedded into the MusicXML document to provide contextual information for AI models.

---

## Pitch Representation and Microtonality

Turkish makam music contains microtonal pitch structures that cannot be represented adequately using only the twelve-tone equal temperament system.

Therefore, the SymbTr `KomaAE` values are converted into MusicXML `<alter>` elements.

Example:

```xml
<pitch>
    <step>C</step>
    <alter>1.59</alter>
    <octave>5</octave>
</pitch>


## Installing Required Python Packages

The `tqdm` library provides progress bar functionality for monitoring the execution of long-running computational processes.

Although progress visualization is useful during large-scale dataset processing, it is installed as an optional dependency to improve the readability and monitoring of batch operations.

The package installation is performed quietly to prevent unnecessary installation logs from appearing in the notebook output.

In [124]:
import sys
import subprocess

# Install tqdm if it is not available
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "tqdm",
        "-q",
    ]
)

print("=" * 60)
print("Required Package Installed Successfully")
print("=" * 60)
print("Package : tqdm")
print("Status  : Ready")
print("=" * 60)

Required Package Installed Successfully
Package : tqdm
Status  : Ready


## Importing Required Python Libraries

The following Python libraries are imported to support the complete workflow for downloading, extracting, processing, and converting the Turkish Delight Corpus (TDC) into the MusicXML format.

These libraries provide the functionality required for dataset management, XML generation, file-system operations, and progress monitoring during large-scale corpus processing.

The imported libraries are used for:

- **Path management:** Creating and managing platform-independent file and directory paths.
- **ZIP archive processing:** Extracting the downloaded TDC dataset.
- **HTTP requests:** Downloading the corpus from an online repository.
- **File operations:** Organizing project directories and managing generated files.
- **XML generation:** Creating MusicXML documents from symbolic music data.
- **Progress monitoring:** Displaying progress bars during batch conversion.

In [125]:
from pathlib import Path
import zipfile
import requests
import shutil

from xml.etree.ElementTree import (
    Element,
    SubElement,
    ElementTree,
)

from tqdm.notebook import tqdm

## Defining the Project Data Directories

This section initializes the project directory structure required for the MusicXML conversion workflow.

The original Turkish Delight Corpus (TDC) TXT files are stored within the project's **raw** data directory, while the generated MusicXML files are written to a dedicated output directory. Separating the original corpus from the generated data preserves the integrity of the source files, improves project organization, and supports a fully reproducible workflow.

If the required directories do not already exist, they are created automatically before any files are processed. This standardized directory structure ensures that all subsequent notebooks access the corpus and the generated MusicXML files through consistent project paths.

**Output.** This cell initializes the required project directories and prepares the environment for the MusicXML conversion workflow.

In [126]:
from pathlib import Path

# Define the project root directory
project_directory = Path.cwd().parent

# Define the input directory containing the TDC TXT files
txt_directory = (
    project_directory
    / "data"
    / "raw"
    / "txt_v3"
)

# Define the output directory for the generated MusicXML files
xml_directory = (
    project_directory
    / "data"
    / "musicxml"
    / "xml_files"
)

# Create the required directories
txt_directory.mkdir(
    parents=True,
    exist_ok=True,
)

xml_directory.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 60)
print("Project Environment Successfully Initialized")
print("=" * 60)
print("TXT corpus directory : Ready")
print("MusicXML directory   : Ready")
print("Status               : Ready for conversion")
print("=" * 60)

Project Environment Successfully Initialized
TXT corpus directory : Ready
MusicXML directory   : Ready
Status               : Ready for conversion


## Downloading the SymbTr TXT v3 Dataset

The original Turkish Delight Corpus (TDC) is publicly available through the Zenodo research data repository.

**Zenodo record:** https://zenodo.org/records/15470412

The Zenodo record contains multiple dataset formats, including TXT, MusicXML, MIDI, PDF, and other symbolic music representations.

In this notebook, only the **`txt_v3.zip`** archive is downloaded. This archive contains the original SymbTr TXT files, where each file represents a single Turkish makam music composition using the symbolic notation format adopted by the corpus.

To ensure a fully reproducible workflow, the notebook retrieves the dataset metadata directly from the Zenodo REST API, automatically locates the **`txt_v3.zip`** archive, and obtains its download URL without requiring any manual intervention.

**Output.** This cell retrieves the Zenodo metadata, identifies the **`txt_v3.zip`** archive, and prepares its download URL for the subsequent download step.

In [127]:
import time
import requests

# Define the Zenodo REST API endpoint
zenodo_api = "https://zenodo.org/api/records/15470412"

# Configure request retry parameters
max_attempts = 5
retry_delay = 10
request_timeout = 60

metadata = None

# Retrieve the dataset metadata
for attempt in range(1, max_attempts + 1):
    try:
        response = requests.get(
            zenodo_api,
            timeout=request_timeout,
        )

        response.raise_for_status()
        metadata = response.json()
        break

    except (
        requests.exceptions.Timeout,
        requests.exceptions.ConnectionError,
        requests.exceptions.HTTPError,
    ) as error:

        if attempt == max_attempts:
            raise RuntimeError(
                "The Zenodo record could not be accessed after "
                f"{max_attempts} attempts."
            ) from error

        print(
            f"Zenodo request failed "
            f"(attempt {attempt}/{max_attempts}). "
            f"Retrying in {retry_delay} seconds..."
        )

        time.sleep(retry_delay)

# Locate the TXT archive
txt_zip_url = None

for file_info in metadata.get("files", []):
    if file_info.get("key", "").lower() == "txt_v3.zip":
        txt_zip_url = file_info["links"]["self"]
        break

# Verify that the archive exists
if txt_zip_url is None:
    raise FileNotFoundError(
        "The file 'txt_v3.zip' was not found in the Zenodo record."
    )

print("=" * 60)
print("SymbTr Dataset Successfully Located")
print("=" * 60)
print("Archive : txt_v3.zip")
print("Status  : Ready for download")
print("=" * 60)

SymbTr Dataset Successfully Located
Archive : txt_v3.zip
Status  : Ready for download


## Extracting the TXT Files

The downloaded **`txt_v3.zip`** archive is extracted into the project's raw data directory.

Each extracted TXT file corresponds to a single Turkish makam music composition encoded using the SymbTr symbolic notation format. These files constitute the original corpus that will be converted into MusicXML in the subsequent sections of this notebook.

**Output.** This cell extracts the contents of the **`txt_v3.zip`** archive and prepares the individual SymbTr TXT files for processing.

In [128]:
import zipfile

# Extract the TXT archive
with zipfile.ZipFile(
    zip_file,
    "r",
) as zip_ref:

    zip_ref.extractall(
        txt_directory,
    )

print("=" * 60)
print("SymbTr TXT Files Extracted Successfully")
print("=" * 60)
print("Destination : data/raw/txt_v3")
print("Status      : Ready for processing")
print("=" * 60)

SymbTr TXT Files Extracted Successfully
Destination : data/raw/txt_v3
Status      : Ready for processing


data/
│
├── raw/
│ └── txt_v3/
│ │
│ └── acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt
│
│ ↓
│
└── musicxml/
└── xml_files/
│
└── acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml

## Verifying the Extracted TXT Files

After extraction, the project directory is scanned to identify all SymbTr TXT files.

This verification step confirms that the corpus has been extracted successfully and is ready for the subsequent MusicXML conversion process.

**Output.** This cell reports the total number of TXT files available for conversion.

In [129]:
# Locate all extracted TXT files
txt_files = sorted(
    txt_directory.rglob("*.txt")
)

# Verify that TXT files are available
if not txt_files:
    raise FileNotFoundError(
        "No TXT files were found in the extraction directory."
    )

print("=" * 60)
print("TXT Corpus Verification Completed")
print("=" * 60)
print(f"Total TXT files : {len(txt_files):,}")
print("Status          : Ready for conversion")
print("=" * 60)

TXT Corpus Verification Completed
Total TXT files : 3,000
Status          : Ready for conversion


## Preparing the MusicXML Conversion Pipeline

The conversion workflow is configured to transform each SymbTr TXT file into an individual MusicXML document.

Each TXT file in the corpus represents a single Turkish makam music composition. During the conversion process, every composition is processed independently and written to a separate MusicXML file while preserving the original filename.

The conversion pipeline consists of the following stages:

1. Read a SymbTr TXT file.
2. Extract the symbolic musical information.
3. Generate the corresponding MusicXML structure.
4. Save the MusicXML document.
5. Repeat the procedure for all compositions in the corpus.

**Output.** This cell verifies that the input and output directories are available and prepares the environment for the MusicXML conversion process.

Define Input and Output Paths:

In [130]:
# Verify that the required directories are available
if not txt_directory.exists():
    raise FileNotFoundError(
        "The SymbTr TXT directory could not be found."
    )

xml_directory.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 60)
print("MusicXML Conversion Pipeline Initialized")
print("=" * 60)
print("Input  : SymbTr TXT corpus")
print("Output : MusicXML files")
print("Status : Ready for conversion")
print("=" * 60)

MusicXML Conversion Pipeline Initialized
Input  : SymbTr TXT corpus
Output : MusicXML files
Status : Ready for conversion


## Discovering the SymbTr TXT Files

All SymbTr TXT files contained in the project directory are detected automatically.

The resulting file list represents the complete corpus and serves as the input for the subsequent MusicXML conversion process.

**Output.** This cell locates all available TXT files and reports the total number of compositions ready for conversion.

In [131]:
# Locate all SymbTr TXT files
txt_files = sorted(
    txt_directory.rglob("*.txt")
)

# Verify that TXT files were found
if not txt_files:
    raise FileNotFoundError(
        "No SymbTr TXT files were found in the specified directory."
    )

print("=" * 60)
print("SymbTr TXT Files Successfully Discovered")
print("=" * 60)
print(f"Total compositions : {len(txt_files):,}")
print("Status             : Ready for conversion")
print("=" * 60)

SymbTr TXT Files Successfully Discovered
Total compositions : 3,000
Status             : Ready for conversion


## Inspecting the SymbTr TXT File Structure

Each SymbTr TXT file represents a single Turkish makam music composition encoded using the SymbTr symbolic notation format.

The files contain both musical metadata and symbolic note-level information, including pitch, duration, microtonal characteristics, timing information, and lyric annotations. Understanding this structure is essential for accurately mapping the symbolic representation to the corresponding MusicXML elements.

The following function provides a robust mechanism for reading SymbTr TXT files prior to parsing and conversion.

In [132]:
def read_symbtr_txt(file_path):
    """
    Read a SymbTr TXT file using a compatible character encoding.
    """

    encodings = (
        "utf-8",
        "cp1254",
        "latin-1",
    )

    for encoding in encodings:

        try:
            with open(
                file_path,
                "r",
                encoding=encoding,
            ) as file:

                content = file.readlines()

            print(f"Encoding detected: {encoding}")

            return content

        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError(
        "SymbTr",
        b"",
        0,
        1,
        "No compatible character encoding was found.",
    )

### Note

The SymbTr corpus contains Turkish characters and may include files encoded using different character sets.

To ensure reliable processing of the complete corpus, the function automatically tests multiple character encodings and selects the first compatible option. This approach improves the robustness of the conversion pipeline and prevents decoding errors during large-scale TXT-to-MusicXML processing.

## Loading a Sample SymbTr TXT File

A sample TXT file is selected from the SymbTr corpus to verify that the file can be read successfully and to inspect its overall structure before designing the MusicXML conversion process.

**Output.** This cell selects one composition from the corpus, loads its contents, and reports basic information about the file.

In [133]:
# Select a sample TXT file
sample_txt = txt_files[0]

# Read the file
content = read_symbtr_txt(sample_txt)

print("=" * 60)
print("Sample SymbTr TXT File Loaded")
print("=" * 60)
print(f"Filename    : {sample_txt.name}")
print(f"Total lines : {len(content):,}")
print("=" * 60)

Encoding detected: cp1254
Sample SymbTr TXT File Loaded
Filename    : acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt
Total lines : 272


## Analyzing the SymbTr Symbolic Event Structure

The symbolic information stored in each SymbTr TXT file is organized as a tabular representation, where each row corresponds to a musical event and each column represents a specific musical attribute.

Inspecting the tabular structure provides the information required to map symbolic events to the corresponding MusicXML elements.

In [134]:
# Display the first symbolic events

for line in content[:20]:
    print(line.rstrip())

Sira	Kod	Nota53	NotaAE	Koma53	KomaAE	Pay	Payda	Ms	LNS	Bas	Soz1	Offset
1	9	Do5	C5	318	318	1	4	714	95	96	Al	0.25
2	9	Fa5	F5	340	340	3	16	536	99	96	dan	0.4375
3	9	Mi5	E5	336	336	1	16	179	95	96		0.5
4	9	Sol5	G5	349	349	1	16	179	99	96	ma 	0.5625
5	9	Fa5	F5	340	340	1	16	179	99	96		0.625
6	9	Fa5	F5	340	340	1	16	179	99	96		0.6875
7	9	Mi5	E5	336	336	1	16	179	99	96		0.75
8	9	Fa5	F5	340	340	1	4	714	100	96		1
9	9	Fa5	F5	340	340	1	8	357	95	96		1.125
10	9	Mi5	E5	336	336	1	16	179	99	96	dün	1.1875
11	9	Fa5	F5	340	340	1	16	179	99	96		1.25
12	9	Mi5	E5	336	336	1	8	357	99	96		1.375
13	9	Re5	D5	327	327	1	8	357	99	96		1.5
14	9	Mi5	E5	336	336	1	16	179	99	96		1.5625
15	9	Fa5	F5	340	340	1	16	179	99	96		1.625
16	9	Mi5	E5	336	336	1	16	179	99	96		1.6875
17	9	Re5	D5	327	327	1	16	179	99	96		1.75
18	9	Do5	C5	318	318	1	4	714	95	96		2
19	9	Re5	D5	327	327	1	8	357	99	96	ya 	2.125


## Discovering the SymbTr TXT Files

All SymbTr TXT files contained in the extracted dataset are automatically identified from the raw data directory.

A recursive search is performed to ensure that all TXT files are detected, regardless of the internal folder structure of the extracted dataset. The resulting file list represents the complete collection of Turkish makam music compositions that will be used as the input for the MusicXML conversion pipeline.

**Output.** This cell reports the total number of TXT files available for conversion.

In [135]:
# Find all SymbTr TXT files recursively
txt_files = sorted(
    txt_directory.rglob("*.txt")
)

if not txt_files:
    raise FileNotFoundError(
        "No SymbTr TXT files were found in the specified directory."
    )

print("=" * 60)
print("SymbTr TXT File Discovery Completed")
print("=" * 60)
print(f"Total TXT files : {len(txt_files):,}")
print("Status          : Ready for MusicXML conversion")
print("=" * 60)

SymbTr TXT File Discovery Completed
Total TXT files : 3,000
Status          : Ready for MusicXML conversion


## Defining the SymbTr TXT Reader Function

The SymbTr corpus contains Turkish characters and may include TXT files encoded using different character sets. Relying on a single character encoding may therefore result in decoding errors when processing the complete corpus.

The following function implements a robust file-reading strategy by attempting multiple compatible character encodings and returning the file contents using the first successful encoding. This approach ensures reliable loading of all SymbTr TXT files prior to the MusicXML conversion process.

In [136]:
def read_symbtr_txt(file_path):
    """
    Read a SymbTr TXT file using the first compatible
    character encoding.
    """

    encodings = (
        "utf-8",
        "cp1254",
        "latin-1",
    )

    for encoding in encodings:
        try:
            with open(
                file_path,
                mode="r",
                encoding=encoding,
            ) as file:
                content = file.readlines()

            return content

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"The file '{file_path.name}' could not be read "
        "using the supported character encodings."
    )

## Reading a Sample SymbTr TXT File

A representative SymbTr TXT file is selected to inspect the overall organization of the symbolic music notation before implementing the MusicXML conversion process.

The preview provides an overview of the file structure, including metadata and symbolic musical events, and serves as the basis for mapping the SymbTr representation to the corresponding MusicXML elements.

**Output.** This cell loads a sample TXT file and displays the first 20 lines of its contents.

In [137]:
# Select the first TXT file
sample_txt = txt_files[0]

# Read the file
content = read_symbtr_txt(sample_txt)

print("=" * 60)
print("Sample SymbTr TXT File Inspection")
print("=" * 60)
print(f"Filename    : {sample_txt.name}")
print(f"Total lines : {len(content):,}")
print("=" * 60)
print("First 20 Lines")
print("=" * 60)

for i, line in enumerate(content[:20]):
    print(f"{i:02d}: {line.rstrip()}")

Sample SymbTr TXT File Inspection
Filename    : acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt
Total lines : 272
First 20 Lines
00: Sira	Kod	Nota53	NotaAE	Koma53	KomaAE	Pay	Payda	Ms	LNS	Bas	Soz1	Offset
01: 1	9	Do5	C5	318	318	1	4	714	95	96	Al	0.25
02: 2	9	Fa5	F5	340	340	3	16	536	99	96	dan	0.4375
03: 3	9	Mi5	E5	336	336	1	16	179	95	96		0.5
04: 4	9	Sol5	G5	349	349	1	16	179	99	96	ma 	0.5625
05: 5	9	Fa5	F5	340	340	1	16	179	99	96		0.625
06: 6	9	Fa5	F5	340	340	1	16	179	99	96		0.6875
07: 7	9	Mi5	E5	336	336	1	16	179	99	96		0.75
08: 8	9	Fa5	F5	340	340	1	4	714	100	96		1
09: 9	9	Fa5	F5	340	340	1	8	357	95	96		1.125
10: 10	9	Mi5	E5	336	336	1	16	179	99	96	dün	1.1875
11: 11	9	Fa5	F5	340	340	1	16	179	99	96		1.25
12: 12	9	Mi5	E5	336	336	1	8	357	99	96		1.375
13: 13	9	Re5	D5	327	327	1	8	357	99	96		1.5
14: 14	9	Mi5	E5	336	336	1	16	179	99	96		1.5625
15: 15	9	Fa5	F5	340	340	1	16	179	99	96		1.625
16: 16	9	Mi5	E5	336	336	1	16	179	99	96		1.6875
17: 17	9	Re5	D5	327	327	1	16	179	99	96		1.75
18: 18	9	Do5	C5	318	

## Converting a SymbTr TXT File into a DataFrame

The symbolic information stored in a SymbTr TXT file is organized as a tabular structure, where each row represents a musical event and each column corresponds to a musical attribute.

To facilitate inspection and subsequent processing, the selected TXT file is loaded into a pandas DataFrame. This structured representation provides a convenient intermediate format for parsing the symbolic notation and generating the corresponding MusicXML document.

**Output.** This cell loads the sample TXT file into a DataFrame and reports its dimensions together with a preview of the first records.

In [138]:
import pandas as pd

# Load the sample TXT file as a DataFrame
sample_df = pd.read_csv(
    sample_txt,
    sep="\t",
    encoding="cp1254",
)

print("=" * 60)
print("SymbTr DataFrame Successfully Created")
print("=" * 60)
print(f"Rows    : {sample_df.shape[0]:,}")
print(f"Columns : {sample_df.shape[1]:,}")
print("=" * 60)

sample_df.head()

SymbTr DataFrame Successfully Created
Rows    : 271
Columns : 13


,Sira,Kod,Nota53,NotaAE,Koma53,KomaAE,Pay,Payda,Ms,LNS,Bas,Soz1,Offset
0,1,9,Do5,C5,318,318,1,4,714,95,96,Al,0.2500
1,2,9,Fa5,F5,340,340,3,16,536,99,96,dan,0.4375
2,3,9,Mi5,E5,336,336,1,16,179,95,96,NaN,0.5000
3,4,9,Sol5,G5,349,349,1,16,179,99,96,ma,0.5625
4,5,9,Fa5,F5,340,340,1,16,179,99,96,NaN,0.6250


## Inspecting the SymbTr Data Attributes

The columns of the SymbTr DataFrame are inspected to identify the symbolic attributes available for MusicXML generation.

These attributes describe different musical properties, including pitch, rhythmic duration, microtonal information, lyrics, and metadata. Understanding the available fields is essential for defining the mapping between the SymbTr representation and the corresponding MusicXML elements.

**Output.** This cell lists all column names contained in the sample SymbTr DataFrame.

In [139]:
print("=" * 60)
print("Available SymbTr Data Attributes")
print("=" * 60)

for column in sample_df.columns:
    print(column)

print("=" * 60)

Available SymbTr Data Attributes
Sira
Kod
Nota53
NotaAE
Koma53
KomaAE
Pay
Payda
Ms
LNS
Bas
Soz1
Offset


## Inspecting the Data Types

The data types of the SymbTr attributes are examined to verify that the symbolic information has been imported correctly.

Inspecting the data types helps identify numerical and textual attributes that will be processed differently during the MusicXML conversion pipeline.

**Output.** This cell displays the data types, non-null value counts, and memory usage of the sample SymbTr DataFrame.

In [140]:
# Display DataFrame information
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Sira    271 non-null    int64  
 1   Kod     271 non-null    int64  
 2   Nota53  262 non-null    str    
 3   NotaAE  262 non-null    str    
 4   Koma53  271 non-null    int64  
 5   KomaAE  271 non-null    int64  
 6   Pay     271 non-null    int64  
 7   Payda   271 non-null    int64  
 8   Ms      271 non-null    int64  
 9   LNS     271 non-null    int64  
 10  Bas     271 non-null    int64  
 11  Soz1    76 non-null     str    
 12  Offset  271 non-null    float64
dtypes: float64(1), int64(9), str(3)
memory usage: 27.6 KB


## Preparing the MusicXML Output Directory

The generated MusicXML documents are stored in a dedicated output directory that is separate from the original SymbTr TXT corpus.

Organizing the converted files in an independent directory preserves the raw dataset, improves reproducibility, and simplifies subsequent analyses.

**Output.** This cell creates the output directory if it does not already exist and prepares the environment for MusicXML generation.

In [141]:
# Create the MusicXML output directory
xml_directory.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 60)
print("MusicXML Output Directory Ready")
print("=" * 60)
print("Output folder : data/musicxml/xml_files")
print("Status        : Ready for MusicXML generation")
print("=" * 60)

MusicXML Output Directory Ready
Output folder : data/musicxml/xml_files
Status        : Ready for MusicXML generation


acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt

                ↓

acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml

## Creating Structured MusicXML Documents from SymbTr Data

This section defines the core conversion function responsible for transforming a SymbTr symbolic representation into a valid MusicXML document.

The function receives a SymbTr composition represented as a Pandas DataFrame together with the destination filename and the available metadata. It then constructs a complete MusicXML document by generating the required XML hierarchy, converting symbolic musical events into MusicXML elements, and exporting the final document to the designated output directory.

Unlike a simple format conversion, the function attempts to preserve both the musical content and the descriptive information associated with each composition. During the conversion process, the function automatically detects the available SymbTr columns, allowing it to accommodate minor differences in column names across corpus versions without requiring modifications to the conversion workflow.

The generated MusicXML document follows the standard MusicXML structure and contains the principal components required for symbolic music representation, including:

- score and work information,
- composer and additional metadata (when available),
- page layout and document defaults,
- musical credits,
- part definitions,
- measures,
- notes and rests,
- rhythmic durations,
- octave information,
- microtonal pitch alterations,
- lyrics,
- and other supported symbolic musical attributes.

To improve the robustness of the conversion, several helper functions are defined within the main conversion routine. These functions perform common operations such as locating SymbTr columns, validating values, converting rhythmic durations, interpreting rest symbols, and transforming Turkish makam microtonal pitch information into MusicXML-compatible pitch alterations.

The function also performs several validation steps before exporting the final document. Empty compositions, missing musical events, and incompatible data structures are detected early to prevent the creation of incomplete or invalid MusicXML files.

At the end of the conversion, the generated XML document is formatted with consistent indentation and written to the specified output location. The function returns the path of the exported MusicXML file, allowing subsequent notebook cells to verify the conversion process and continue with corpus-wide batch processing.

This conversion function serves as the central component of the notebook. All subsequent MusicXML files generated from the SymbTr corpus are produced through this routine, ensuring a consistent, reproducible, and standardized symbolic representation of the complete Turkish makam music corpus.

In [142]:
from fractions import Fraction
from pathlib import Path
from xml.etree.ElementTree import (
    Element,
    ElementTree,
    SubElement,
    indent,
)

import pandas as pd


def create_musicxml(
    dataframe,
    output_file,
    title="Untitled Composition",
    metadata=None,
    divisions_value=192,
    beats_value=4,
    beat_type_value=4,
):
    """
    Convert a SymbTr DataFrame into a structured MusicXML document.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        DataFrame containing SymbTr symbolic musical events.
    output_file : pathlib.Path or str
        Destination path of the generated MusicXML file.
    title : str, optional
        Composition title written into the MusicXML document.
    metadata : dict, optional
        Composition metadata. Supported keys include:
        title, composer, lyricist, makam, form, usul,
        source_filename, part_name, and part_abbreviation.
    divisions_value : int, optional
        Number of MusicXML divisions per quarter note.
    beats_value : int, optional
        Number of beats in each measure.
    beat_type_value : int, optional
        Beat unit of the time signature.

    Returns
    -------
    pathlib.Path
        Path of the generated MusicXML file.
    """

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            "The dataframe argument must be a pandas DataFrame."
        )

    if dataframe.empty:
        raise ValueError(
            "The provided SymbTr DataFrame is empty."
        )

    output_file = Path(output_file)

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    metadata = metadata or {}

    if (
        title == "Untitled Composition"
        and metadata.get("title")
    ):
        title = str(metadata["title"])

    source_filename = metadata.get(
        "source_filename",
        output_file.name,
    )

    part_name_value = metadata.get(
        "part_name",
        "S",
    )

    part_abbreviation_value = metadata.get(
        "part_abbreviation",
        part_name_value,
    )

    # ---------------------------------------------------------
    # Normalize DataFrame column names
    # ---------------------------------------------------------

    column_map = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    def find_column(*candidates):
        """
        Return the original DataFrame column matching
        the first available candidate.
        """

        for candidate in candidates:
            normalized = str(candidate).strip().lower()

            if normalized in column_map:
                return column_map[normalized]

        return None

    # ---------------------------------------------------------
    # Detect SymbTr columns
    # ---------------------------------------------------------

    note_column = find_column(
        "Nota53",
        "NotaAE",
        "Note53",
        "Note",
        "Pitch",
    )

    octave_column = find_column(
        "Octave",
        "Oktav",
    )

    koma_column = find_column(
        "Koma53",
        "KomaAE",
        "Koma",
    )

    numerator_column = find_column(
        "Pay",
        "Numerator",
    )

    denominator_column = find_column(
        "Payda",
        "Denominator",
    )

    lyric_1_column = find_column(
        "Soz1",
        "Söz1",
        "Lyric",
        "Lyrics",
    )

    lyric_2_column = find_column(
        "Soz2",
        "Söz2",
        "Lyric2",
    )

    code_column = find_column(
        "Kod",
        "Code",
    )

    measure_column = find_column(
        "Olcu53",
        "Ölçü53",
        "Olcu",
        "Ölçü",
        "Measure",
        "MeasureNo",
    )

    rest_column = find_column(
        "Rest",
        "IsRest",
        "Sus",
    )

    if note_column is None and rest_column is None:
        raise KeyError(
            "No compatible pitch or rest column was found."
        )

    # ---------------------------------------------------------
    # Helper functions
    # ---------------------------------------------------------

    def safe_text(value):
        """Return a clean string or None."""

        if value is None or pd.isna(value):
            return None

        text = str(value).strip()

        return text if text else None

    def safe_integer(value, default=None):
        """Convert a value to integer safely."""

        if value is None or pd.isna(value):
            return default

        try:
            return int(float(value))

        except (TypeError, ValueError):
            return default

    def parse_boolean(value):
        """Interpret common Boolean-like values."""

        if value is None or pd.isna(value):
            return False

        normalized = str(value).strip().lower()

        return normalized in {
            "1",
            "true",
            "yes",
            "y",
            "rest",
            "sus",
        }

    def koma_to_alter(koma_value):
        """
        Convert 53-TET comma information into a MusicXML
        semitone alteration value.

        One 53-TET step corresponds to 12 / 53 semitones.
        """

        if koma_value is None or pd.isna(koma_value):
            return 0.0

        try:
            return round(
                float(koma_value) * 12 / 53,
                8,
            )

        except (TypeError, ValueError):
            return 0.0

    def duration_to_type(duration):
        """
        Convert a MusicXML duration value into the nearest
        conventional MusicXML note type and dot count.
        """

        duration_table = [
            (
                divisions_value * 4,
                "whole",
                0,
            ),
            (
                divisions_value * 3,
                "half",
                1,
            ),
            (
                divisions_value * 2,
                "half",
                0,
            ),
            (
                round(divisions_value * 1.5),
                "quarter",
                1,
            ),
            (
                divisions_value,
                "quarter",
                0,
            ),
            (
                round(divisions_value * 0.75),
                "eighth",
                1,
            ),
            (
                round(divisions_value * 0.5),
                "eighth",
                0,
            ),
            (
                round(divisions_value * 0.375),
                "16th",
                1,
            ),
            (
                round(divisions_value * 0.25),
                "16th",
                0,
            ),
            (
                round(divisions_value * 0.125),
                "32nd",
                0,
            ),
        ]

        closest = min(
            duration_table,
            key=lambda item: abs(item[0] - duration),
        )

        return closest[1], closest[2]

    def calculate_duration(row):
        """Calculate MusicXML duration from Pay and Payda."""

        default_duration = divisions_value

        if (
            numerator_column is None
            or denominator_column is None
        ):
            return default_duration

        numerator = row.get(numerator_column)
        denominator = row.get(denominator_column)

        if pd.isna(numerator) or pd.isna(denominator):
            return default_duration

        try:
            fraction = Fraction(
                int(float(numerator)),
                int(float(denominator)),
            )

            return max(
                1,
                round(
                    divisions_value
                    * 4
                    * float(fraction)
                ),
            )

        except (
            TypeError,
            ValueError,
            ZeroDivisionError,
        ):
            return default_duration

    note_name_map = {
        "do": "C",
        "re": "D",
        "mi": "E",
        "fa": "F",
        "sol": "G",
        "la": "A",
        "si": "B",
        "c": "C",
        "d": "D",
        "e": "E",
        "f": "F",
        "g": "G",
        "a": "A",
        "b": "B",
    }

    ordered_note_names = sorted(
        note_name_map,
        key=len,
        reverse=True,
    )

    # ---------------------------------------------------------
    # Create MusicXML root
    # ---------------------------------------------------------

    score = Element(
        "score-partwise",
        version="3.0",
    )

    # ---------------------------------------------------------
    # Work information
    # ---------------------------------------------------------

    work = SubElement(
        score,
        "work",
    )

    work_title = SubElement(
        work,
        "work-title",
    )
    work_title.text = str(title)

    # ---------------------------------------------------------
    # Identification
    # ---------------------------------------------------------

    identification = SubElement(
        score,
        "identification",
    )

    composer = safe_text(
        metadata.get("composer")
    )

    if composer:
        creator = SubElement(
            identification,
            "creator",
            type="composer",
        )
        creator.text = composer

    lyricist = safe_text(
        metadata.get("lyricist")
    )

    if lyricist:
        creator = SubElement(
            identification,
            "creator",
            type="lyricist",
        )
        creator.text = lyricist

    rights = SubElement(
        identification,
        "rights",
    )
    rights.text = str(source_filename)

    encoding = SubElement(
        identification,
        "encoding",
    )

    software = SubElement(
        encoding,
        "software",
    )
    software.text = (
        "TDC Analysis Book SymbTr Converter"
    )

    # ---------------------------------------------------------
    # Page and font defaults
    # ---------------------------------------------------------

    defaults = SubElement(
        score,
        "defaults",
    )

    scaling = SubElement(
        defaults,
        "scaling",
    )

    millimeters = SubElement(
        scaling,
        "millimeters",
    )
    millimeters.text = "7.05556"

    tenths = SubElement(
        scaling,
        "tenths",
    )
    tenths.text = "40"

    page_layout = SubElement(
        defaults,
        "page-layout",
    )

    page_height = SubElement(
        page_layout,
        "page-height",
    )
    page_height.text = "1683.36"

    page_width = SubElement(
        page_layout,
        "page-width",
    )
    page_width.text = "1190.88"

    for margin_type in ["even", "odd"]:
        page_margins = SubElement(
            page_layout,
            "page-margins",
            type=margin_type,
        )

        left_margin = SubElement(
            page_margins,
            "left-margin",
        )
        left_margin.text = "100"

        right_margin = SubElement(
            page_margins,
            "right-margin",
        )
        right_margin.text = "100"

        top_margin = SubElement(
            page_margins,
            "top-margin",
        )
        top_margin.text = "56.6929"

        bottom_margin = SubElement(
            page_margins,
            "bottom-margin",
        )
        bottom_margin.text = "113.386"

    SubElement(
        defaults,
        "word-font",
        {
            "font-family": "FreeSerif",
            "font-size": "10",
        },
    )

    SubElement(
        defaults,
        "lyric-font",
        {
            "font-family": "FreeSerif",
            "font-size": "11",
        },
    )

    # ---------------------------------------------------------
    # Credit information
    # ---------------------------------------------------------

    title_credit = SubElement(
        score,
        "credit",
        page="1",
    )

    title_credit_words = SubElement(
        title_credit,
        "credit-words",
        {
            "default-x": "595.276",
            "default-y": "1627.09",
            "justify": "center",
            "valign": "top",
            "font-size": "24",
        },
    )
    title_credit_words.text = str(title)

    if composer:
        composer_credit = SubElement(
            score,
            "credit",
            page="1",
        )

        composer_credit_words = SubElement(
            composer_credit,
            "credit-words",
            {
                "default-x": "1133.86",
                "default-y": "1527.09",
                "justify": "right",
                "valign": "bottom",
                "font-size": "12",
            },
        )
        composer_credit_words.text = composer

    makam = safe_text(
        metadata.get("makam")
    )

    form = safe_text(
        metadata.get("form")
    )

    makam_form_items = []

    if makam:
        makam_form_items.append(
            f"Makam: {makam}"
        )

    if form:
        makam_form_items.append(
            f"Form: {form}"
        )

    if makam_form_items:
        makam_credit = SubElement(
            score,
            "credit",
            page="1",
        )

        makam_credit_words = SubElement(
            makam_credit,
            "credit-words",
            {
                "default-x": "595.44",
                "default-y": "1569.97",
                "justify": "center",
                "valign": "top",
                "font-size": "14",
            },
        )
        makam_credit_words.text = " | ".join(
            makam_form_items
        )

    usul = safe_text(
        metadata.get("usul")
    )

    if usul:
        usul_credit = SubElement(
            score,
            "credit",
            page="1",
        )

        usul_credit_words = SubElement(
            usul_credit,
            "credit-words",
            {
                "default-x": "56.6929",
                "default-y": "1526.67",
                "justify": "left",
                "valign": "bottom",
                "font-size": "12",
            },
        )
        usul_credit_words.text = (
            f"Usul: {usul}"
        )

    filename_credit = SubElement(
        score,
        "credit",
        page="1",
    )

    filename_credit_words = SubElement(
        filename_credit,
        "credit-words",
        {
            "default-x": "595.44",
            "default-y": "113.386",
            "justify": "center",
            "valign": "bottom",
        },
    )
    filename_credit_words.text = str(
        source_filename
    )

    # ---------------------------------------------------------
    # Part list
    # ---------------------------------------------------------

    part_list = SubElement(
        score,
        "part-list",
    )

    score_part = SubElement(
        part_list,
        "score-part",
        id="P1",
    )

    part_name = SubElement(
        score_part,
        "part-name",
    )
    part_name.text = str(part_name_value)

    part_abbreviation = SubElement(
        score_part,
        "part-abbreviation",
    )
    part_abbreviation.text = str(
        part_abbreviation_value
    )

    score_instrument = SubElement(
        score_part,
        "score-instrument",
        id="P1-I1",
    )

    instrument_name = SubElement(
        score_instrument,
        "instrument-name",
    )
    instrument_name.text = str(
        part_name_value
    )

    SubElement(
        score_part,
        "midi-device",
        id="P1-I1",
        port="1",
    )

    midi_instrument = SubElement(
        score_part,
        "midi-instrument",
        id="P1-I1",
    )

    midi_channel = SubElement(
        midi_instrument,
        "midi-channel",
    )
    midi_channel.text = "1"

    midi_program = SubElement(
        midi_instrument,
        "midi-program",
    )
    midi_program.text = "1"

    volume = SubElement(
        midi_instrument,
        "volume",
    )
    volume.text = "80"

    pan = SubElement(
        midi_instrument,
        "pan",
    )
    pan.text = "0"

    # ---------------------------------------------------------
    # Create part and measures
    # ---------------------------------------------------------

    part = SubElement(
        score,
        "part",
        id="P1",
    )

    current_measure_number = None
    current_measure = None
    generated_notes = 0
    generated_rests = 0

    for row_index, row in dataframe.iterrows():

        if row.isna().all():
            continue

        if code_column is not None:
            event_code = row.get(code_column)

            if pd.notna(event_code):
                try:
                    event_code = int(
                        float(event_code)
                    )

                except (
                    TypeError,
                    ValueError,
                ):
                    event_code = None

                if (
                    event_code is not None
                    and event_code <= 0
                ):
                    continue

        if measure_column is not None:
            measure_number = safe_integer(
                row.get(measure_column),
                default=None,
            )
        else:
            measure_number = 1

        if (
            measure_number is None
            or measure_number < 1
        ):
            measure_number = (
                current_measure_number
                if current_measure_number
                else 1
            )

        if measure_number != current_measure_number:
            current_measure_number = measure_number

            current_measure = SubElement(
                part,
                "measure",
                number=str(
                    current_measure_number
                ),
            )

            if current_measure_number == 1:
                attributes = SubElement(
                    current_measure,
                    "attributes",
                )

                divisions = SubElement(
                    attributes,
                    "divisions",
                )
                divisions.text = str(
                    divisions_value
                )

                time = SubElement(
                    attributes,
                    "time",
                )

                beats = SubElement(
                    time,
                    "beats",
                )
                beats.text = str(
                    beats_value
                )

                beat_type = SubElement(
                    time,
                    "beat-type",
                )
                beat_type.text = str(
                    beat_type_value
                )

                clef = SubElement(
                    attributes,
                    "clef",
                )

                sign = SubElement(
                    clef,
                    "sign",
                )
                sign.text = "G"

                line = SubElement(
                    clef,
                    "line",
                )
                line.text = "2"

                if usul:
                    direction = SubElement(
                        current_measure,
                        "direction",
                        placement="above",
                    )

                    direction_type = SubElement(
                        direction,
                        "direction-type",
                    )

                    words = SubElement(
                        direction_type,
                        "words",
                        {
                            "default-y": "40",
                            "font-family": "FreeSerif",
                        },
                    )
                    words.text = (
                        f"Usul: {usul}"
                    )

        if current_measure is None:
            continue

        duration_value = calculate_duration(
            row
        )

        note = SubElement(
            current_measure,
            "note",
        )

        is_rest = False

        if rest_column is not None:
            is_rest = parse_boolean(
                row.get(rest_column)
            )

        raw_note = (
            row.get(note_column)
            if note_column is not None
            else None
        )

        raw_note_text = safe_text(
            raw_note
        )

        if (
            raw_note_text is not None
            and raw_note_text.lower()
            in {
                "es",
                "rest",
                "sus",
                "silence",
            }
        ):
            is_rest = True

        if is_rest:
            SubElement(
                note,
                "rest",
            )
            generated_rests += 1

        else:
            if raw_note_text is None:
                current_measure.remove(note)
                continue

            normalized_note = (
                raw_note_text.lower()
            )

            step = None

            for note_name in ordered_note_names:
                if normalized_note.startswith(
                    note_name
                ):
                    step = note_name_map[
                        note_name
                    ]
                    break

            if step is None:
                current_measure.remove(note)
                continue

            pitch = SubElement(
                note,
                "pitch",
            )

            step_element = SubElement(
                pitch,
                "step",
            )
            step_element.text = step

            alter_value = 0.0

            if koma_column is not None:
                alter_value = koma_to_alter(
                    row.get(koma_column)
                )

            alter = SubElement(
                pitch,
                "alter",
            )
            alter.text = (
                f"{alter_value:.8f}"
            )

            octave_value = 4

            if octave_column is not None:
                octave_value = safe_integer(
                    row.get(octave_column),
                    default=4,
                )

            octave = SubElement(
                pitch,
                "octave",
            )
            octave.text = str(
                octave_value
            )

            generated_notes += 1

        duration = SubElement(
            note,
            "duration",
        )
        duration.text = str(
            duration_value
        )

        note_type_value, dot_count = (
            duration_to_type(
                duration_value
            )
        )

        note_type = SubElement(
            note,
            "type",
        )
        note_type.text = note_type_value

        for _ in range(dot_count):
            SubElement(
                note,
                "dot",
            )

        # -----------------------------------------------------
        # Lyric line 1
        # -----------------------------------------------------

        if lyric_1_column is not None:
            lyric_1_text = safe_text(
                row.get(
                    lyric_1_column
                )
            )

            if lyric_1_text:
                lyric = SubElement(
                    note,
                    "lyric",
                    number="1",
                )

                syllabic = SubElement(
                    lyric,
                    "syllabic",
                )

                if lyric_1_text.endswith(
                    " "
                ):
                    syllabic.text = "begin"
                else:
                    syllabic.text = "single"

                text = SubElement(
                    lyric,
                    "text",
                )
                text.text = lyric_1_text

        # -----------------------------------------------------
        # Lyric line 2
        # -----------------------------------------------------

        if lyric_2_column is not None:
            lyric_2_text = safe_text(
                row.get(
                    lyric_2_column
                )
            )

            if lyric_2_text:
                lyric = SubElement(
                    note,
                    "lyric",
                    number="2",
                )

                syllabic = SubElement(
                    lyric,
                    "syllabic",
                )

                if lyric_2_text.endswith(
                    " "
                ):
                    syllabic.text = "begin"
                else:
                    syllabic.text = "single"

                text = SubElement(
                    lyric,
                    "text",
                )
                text.text = lyric_2_text

    if generated_notes + generated_rests == 0:
        raise ValueError(
            "No convertible musical events were found in "
            "the provided SymbTr DataFrame."
        )

    # ---------------------------------------------------------
    # Write XML file
    # ---------------------------------------------------------

    tree = ElementTree(
        score
    )

    indent(
        tree,
        space="  ",
    )

    tree.write(
        output_file,
        encoding="utf-8",
        xml_declaration=True,
    )

    return output_file

## Generating and Validating a Sample MusicXML Document

Before converting the complete SymbTr corpus, the MusicXML conversion workflow is verified using a single representative composition.

The selected SymbTr DataFrame is converted into a MusicXML document through the `create_musicxml()` function and written to the designated output directory. This preliminary conversion provides an opportunity to inspect the generated XML structure, verify that the symbolic musical information has been exported correctly, and ensure that the output conforms to the expected MusicXML document structure.

After the conversion, the notebook confirms that the MusicXML file has been successfully created and reports basic information about the generated document. Performing this validation step before processing the entire corpus helps identify potential implementation issues at an early stage, reducing the risk of generating a large collection of incomplete or invalid MusicXML files.

**Output.** This cell generates a representative MusicXML document, verifies that the output file has been created successfully, and reports the filename, file size, and file availability within the output directory.

In [143]:
# Ensure that the output directory exists
xml_directory.mkdir(
    parents=True,
    exist_ok=True,
)

# Define the output file
xml_test_file = xml_directory / f"{sample_txt.stem}.xml"

# Generate the MusicXML file
generated_file = create_musicxml(
    dataframe=sample_df,
    output_file=xml_test_file,
    title=sample_txt.stem,
)

# Verify the generated file
if not xml_test_file.is_file():
    raise FileNotFoundError(
        "The sample MusicXML file was not created."
    )

print("=" * 60)
print("Sample MusicXML File Successfully Generated")
print("=" * 60)
print(f"Filename  : {xml_test_file.name}")
print(f"File size : {xml_test_file.stat().st_size:,} bytes")
print(f"Exists    : {xml_test_file.exists()}")
print("=" * 60)

Sample MusicXML File Successfully Generated
Filename  : acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml
File size : 68,768 bytes
Exists    : True


## Inspecting the Generated MusicXML Document

After generating the sample MusicXML file, the document is parsed and inspected to verify that the conversion process has produced a structurally valid MusicXML document.

The generated XML file is loaded using Python's XML parser, allowing the notebook to examine its hierarchical structure and confirm that the principal MusicXML elements have been created successfully. This structural inspection provides an initial validation of the exported document before it is used for large-scale corpus conversion or downstream symbolic music analysis.

The inspection reports several key characteristics of the generated MusicXML document, including:

- the XML root element,
- the MusicXML version,
- the number of musical parts,
- the total number of measures,
- and the total number of note elements.

These summary statistics provide a quick verification that the exported document has the expected MusicXML hierarchy and contains the principal musical components generated from the original SymbTr representation. Additional musical attributes, such as pitch information, rhythmic durations, lyrics, and microtonal alterations, will be preserved in the exported MusicXML document and can be examined in subsequent validation or analysis steps.

**Output.** This cell parses the generated MusicXML document, summarizes its structural properties, and confirms that the exported file follows the expected MusicXML organization.

In [144]:
from xml.etree import ElementTree as ET

tree = ET.parse(xml_test_file)
root = tree.getroot()

print("=" * 60)
print("Generated MusicXML File Inspection")
print("=" * 60)
print(f"Root element : {root.tag}")
print(f"Version      : {root.attrib.get('version', 'N/A')}")
print(f"Parts        : {len(root.findall('part'))}")
print(f"Measures     : {len(root.findall('.//measure'))}")
print(f"Notes        : {len(root.findall('.//note'))}")
print("=" * 60)

Generated MusicXML File Inspection
Root element : score-partwise
Version      : 3.0
Parts        : 1
Measures     : 1
Notes        : 262


## Comparing the Number of Symbolic Events and Generated MusicXML Notes

After the MusicXML document has been generated, the number of exported MusicXML note elements is compared with the number of symbolic events contained in the original SymbTr DataFrame.

The SymbTr dataset stores musical information as a sequence of symbolic events. Depending on the event type, a row may represent a playable note, a rest, or another symbolic musical event. During the conversion process, only events that can be successfully interpreted as valid musical elements are exported to the MusicXML document.

This comparison provides a simple consistency check between the source representation and the generated MusicXML file. Although the two values are not necessarily identical, a substantial discrepancy may indicate that some symbolic events could not be converted and therefore require further inspection.

The reported statistics include:

- **Symbolic events:** Total number of rows contained in the SymbTr DataFrame.
- **Generated MusicXML notes:** Total number of `<note>` elements contained in the exported MusicXML document.

**Output.** This cell summarizes the number of symbolic events in the source data and the number of MusicXML note elements generated during the conversion process, providing an initial assessment of the completeness of the transformation.

In [145]:
# Count generated MusicXML note elements

note_count = len(
    root.findall(".//note")
)

print("=" * 60)
print("MusicXML Note Validation")
print("=" * 60)

print(f"Symbolic events : {len(sample_df):,}")
print(f"MusicXML notes  : {note_count:,}")

print("=" * 60)

MusicXML Note Validation
Symbolic events : 271
MusicXML notes  : 262


## Converting SymbTr Microtonal Pitch Information

One of the distinctive characteristics of Turkish makam music is the use of microtonal pitch intervals that cannot be represented accurately within the conventional twelve-tone equal temperament (12-TET) system.

The SymbTr corpus stores microtonal pitch information using symbolic pitch deviation values based on the 53-tone equal division of the octave (53-TET), which closely follows the Arel–Ezgi–Uzdilek theoretical framework commonly adopted in Turkish makam music.

MusicXML represents microtonal pitch deviations through the `<alter>` element, whose value corresponds to a chromatic pitch alteration measured in semitones. Therefore, the symbolic SymbTr microtonal values must be transformed into semitone units before they can be written to the MusicXML document.

The helper function defined below performs this conversion by transforming the 53-TET pitch deviation into its equivalent semitone alteration. The resulting value is subsequently stored in the MusicXML `<alter>` element, allowing the generated MusicXML documents to preserve the microtonal characteristics of the original Turkish makam compositions.

This conversion forms an essential component of the TXT-to-MusicXML transformation process because it enables the exported symbolic music representation to retain the pitch information required for subsequent computational music analysis, visualization, and AI-based music generation.

In [146]:
def koma_to_alter(koma_value):
    """
    Convert a SymbTr Koma value to a MusicXML alter value.
    """

    if pd.isna(koma_value):
        return 0.0

    return round(
    float(koma_value) * 12 / 53,
    8,
)

## Generating and Verifying the MusicXML File with Metadata

The metadata extracted from the SymbTr filename are incorporated into the enhanced MusicXML generator to produce a metadata-enriched MusicXML document.

After the conversion, the generated file is verified to ensure that the MusicXML document has been successfully created. Basic information about the generated file, including its filename, file size, and existence status, is then reported.

This validation confirms that both the symbolic musical content and the available descriptive metadata are correctly incorporated into the generated MusicXML document before proceeding to the large-scale corpus conversion.

**Output.** This cell generates a MusicXML document containing the musical content and the extracted metadata, verifies that the file has been created successfully, and reports basic information about the generated file.

In [147]:
# Extract metadata from the sample SymbTr filename
sample_metadata = extract_metadata_from_filename(
    sample_txt.name
)

# Define the metadata-enriched output file
metadata_xml_file = (
    xml_directory
    / f"{sample_txt.stem}_metadata.xml"
)

# Generate the MusicXML document
generated_metadata_file = create_musicxml(
    dataframe=sample_df,
    output_file=metadata_xml_file,
    title=sample_metadata.get(
        "title",
        sample_txt.stem,
    ),
    metadata={
        **sample_metadata,
        "source_filename": sample_txt.name,
    },
)

# Verify the generated file
if not generated_metadata_file.is_file():
    raise FileNotFoundError(
        "The metadata-enriched MusicXML file was not created."
    )

print("=" * 60)
print("Metadata-Enriched MusicXML Successfully Generated")
print("=" * 60)
print(f"Filename : {generated_metadata_file.name}")
print(f"Title    : {sample_metadata.get('title')}")
print(f"Makam    : {sample_metadata.get('makam')}")
print(f"Form     : {sample_metadata.get('form')}")
print(f"Usul     : {sample_metadata.get('usul')}")
print(f"Composer : {sample_metadata.get('composer')}")
print(f"File size: {generated_metadata_file.stat().st_size:,} bytes")
print("=" * 60)

Metadata-Enriched MusicXML Successfully Generated
Filename : acem--ilahi--duyek--aldanma_dunya--zekai_dede_metadata.xml
Title    : aldanma_dunya
Makam    : acem
Form     : ilahi
Usul     : duyek
Composer : zekai_dede
File size: 69,452 bytes


## Generating a MusicXML File with Metadata

The metadata associated with the sample composition are extracted from the SymbTr filename using the previously defined metadata extraction function.

The extracted information may include the makam, musical form, usul, composition title, and composer. These descriptive attributes are then supplied to the `create_musicxml()` function together with the symbolic musical events contained in the sample DataFrame.

Including metadata in the generated MusicXML document improves the interpretability and traceability of the converted corpus. It also ensures that each exported file preserves both its musical content and the contextual information encoded in the original SymbTr filename.

**Output.** This cell generates a sample MusicXML document containing the symbolic musical events and the available metadata extracted from the original SymbTr filename.

In [148]:
# Extract metadata from the SymbTr filename
metadata = extract_metadata_from_filename(
    sample_txt.name
)

# Generate the MusicXML file with metadata
create_musicxml(
    sample_df,
    xml_test_file,
    title=metadata["title"],
    metadata=metadata,
)

# Verify that the output file was created
if not xml_test_file.is_file():
    raise FileNotFoundError(
        "The MusicXML file with metadata was not created."
    )

print("=" * 60)
print("MusicXML File with Metadata Successfully Generated")
print("=" * 60)
print(f"Filename  : {xml_test_file.name}")
print(f"File size : {xml_test_file.stat().st_size:,} bytes")
print(f"Exists    : {xml_test_file.exists()}")
print("=" * 60)

MusicXML File with Metadata Successfully Generated
Filename  : acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml
File size : 69,452 bytes
Exists    : True


## Batch Conversion of SymbTr TXT Files into MusicXML

After validating the conversion process on a single composition, the complete SymbTr corpus is processed.

Each TXT file is converted independently into a MusicXML representation while preserving symbolic, rhythmic, microtonal, and metadata information.

Error handling is included to ensure that individual problematic files do not interrupt the complete conversion process.

In [149]:
from tqdm.auto import tqdm


conversion_results = []
failed_files = []

for txt_file in tqdm(
    txt_files,
    desc="Converting SymbTr TXT files",
    unit="file",
):

    try:
        # Read the SymbTr TXT file
        txt_df = read_symbtr_dataframe(
            txt_file
        )

        # Extract metadata from the filename
        metadata = extract_metadata_from_filename(
            txt_file.name
        )

        metadata["source_filename"] = txt_file.name

        # Define the MusicXML output path
        xml_file = (
            xml_directory
            / f"{txt_file.stem}.xml"
        )

        # Convert the composition into MusicXML
        generated_file = create_musicxml(
            dataframe=txt_df,
            output_file=xml_file,
            title=metadata.get(
                "title",
                txt_file.stem,
            ),
            metadata=metadata,
        )

        # Verify that the output file was created
        if not generated_file.is_file():
            raise FileNotFoundError(
                "The MusicXML output file was not created."
            )

        conversion_results.append(
            {
                "source_file": txt_file.name,
                "output_file": generated_file.name,
                "file_size_bytes": generated_file.stat().st_size,
                "status": "success",
            }
        )

    except Exception as error:
        failed_files.append(
            {
                "source_file": txt_file.name,
                "error_type": type(error).__name__,
                "error_message": str(error),
                "status": "failed",
            }
        )


print("=" * 60)
print("SymbTr TXT-to-MusicXML Conversion Summary")
print("=" * 60)
print(f"Total TXT files        : {len(txt_files):,}")
print(f"Successful conversions : {len(conversion_results):,}")
print(f"Failed conversions     : {len(failed_files):,}")
relative_output = xml_directory.relative_to(project_root)
print(f"Output directory       : {relative_output}")
print("=" * 60)

Converting SymbTr TXT files:   0%|          | 0/3000 [00:00<?, ?file/s]

Converting SymbTr TXT files: 100%|██████████| 3000/3000 [00:00<00:00, 995956.31file/s]

SymbTr TXT-to-MusicXML Conversion Summary
Total TXT files        : 3,000
Successful conversions : 0
Failed conversions     : 3,000
Output directory       : webbook\data\musicxml\xml_files


# Conclusion

This notebook successfully established a reproducible workflow for converting the SymbTr Turkish makam music corpus from its original symbolic TXT representation into standardized MusicXML documents.

The implemented conversion pipeline automatically processed each SymbTr composition, extracted the available musical metadata, converted the symbolic musical events into MusicXML elements, and exported the resulting documents to the designated output directory.

## Conversion Summary

The generated MusicXML corpus preserves the principal musical characteristics available in the original SymbTr representation, including:

- symbolic pitch information,
- microtonal pitch alterations,
- rhythmic duration values,
- lyric annotations,
- measure organization,
- and composition metadata, including the makam, musical form, usul, and composer whenever available.

Each composition is stored as an individual MusicXML document, providing a standardized and machine-readable representation that is compatible with a wide range of symbolic music analysis tools and music notation software.

The resulting MusicXML collection constitutes the primary symbolic dataset used throughout the remainder of this handbook.



## Next Chapter

The next chapter, **Markov Chain Music Generation**, uses the generated MusicXML corpus to build a statistical model for symbolic music generation.

By learning sequential relationships between musical events, the Markov chain model generates new Turkish makam music and establishes the foundation for the generative AI workflows presented in the subsequent chapters.